# LLMs

Contents:


- How to Use LLMs
  - Local LLM Inference - CPU & GPU
  - Inference via external API
- Key takeaways about LLM through examples
  - Hallucinations
  - LLM Strengths
  - LLM Weaknesses

## 0. Environment setup

Google Colab can run notebooks on CPU or GPU. We will demonstrate both options.

Go to Runtime → Change runtime type → GPU

In [ ]:
import os, shutil

# Core libs (HF + OpenAI)
!pip -q install -U huggingface_hub transformers accelerate sentencepiece openai

# llama-cpp-python:
# - CPU: pip installs a prebuilt wheel (fast)
# - GPU (CUDA): needs a local build with GGML_CUDA enabled
gpu_available = shutil.which("nvidia-smi") is not None
print("GPU runtime detected:", gpu_available)

if gpu_available:
    !nvidia-smi
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    os.environ["FORCE_CMAKE"] = "1"
    # Force a rebuild with CUDA support
    # !pip -q install --no-cache-dir --force-reinstall llama-cpp-python

!pip -q install -U llama-cpp-python


## 1. How to Use LLMs

Two common ways to use LLMs in practice:

1. **Local inference with open-weight models**  
   (small models on CPU or larger models on GPU)

2. **Hosted APIs from model providers:**
   - Yandex Cloud AI Studio  
   - OpenAI  
   - ...

### Trade-offs

- **Local inference →** privacy, full control, predictable infrastructure  
  *(but you are responsible for setup and performance)*

- **API →** fastest path to high quality and scalability  
  *(but you pay per token and depend on a provider)*


---

## 1.1 Local LLM Inference

- **Open weights →** you download a model (e.g., from Hugging Face) and run it locally

- **CPU vs GPU**
  - **CPU →** suitable for very small models (demos, simple tasks, prototyping)
  - **GPU →** enables larger models with much better latency and quality

We will start with a tiny CPU-friendly model, then run a GPU example (if a GPU runtime is enabled).


In [ ]:
# Set parameters and prompts that we will use through notebook

TEMPERATURE = 0.5
MAX_TOKENS = 400
N_CTX = 2048
SYSTEM_PROMPT = "You're my personal assistant. Always start your response by saying 'Hello, Alyona!'"
USER_QUERY = "Give me 3 ideas where AI agents are useful."

PLAIN_PROMPT = f"""\
  System: {SYSTEM_PROMPT}\n
  User: {USER_QUERY}\n
  Assistant:
"""

MESSAGES_PROMPT = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUERY},
]



#### 1.1.1. CPU demo

In [ ]:
# --- 1) Download a tiny Llama-compatible GGUF quantized model ---
from huggingface_hub import hf_hub_download

cpu_model_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
)

print("CPU model path:", cpu_model_path)

In [ ]:
# --- 2) Load with llama.cpp ---
from llama_cpp import Llama

cpu_llm = Llama(
    model_path=cpu_model_path,
    n_ctx=N_CTX,
    n_gpu_layers=0, # CPU only
    verbose=False
)

print("✅ llama model loaded for CPU")

In [ ]:
# --- 3) Inference ---

# --- 3.1) Inference with a single plain prompt ---
def ask_llm_plain_prompt(llm):
  out_plain = llm(
      PLAIN_PROMPT,
      max_tokens=MAX_TOKENS,
      temperature=TEMPERATURE,
  )
  print("\n--- Plain prompt output ---")
  print(out_plain["choices"][0]["text"])


# --- 3.2) Inference with roles via chat completion. That's why roles matter
def ask_llm_chat_prompt(llm):
  out_chat = llm.create_chat_completion(
      messages=MESSAGES_PROMPT,
      temperature=TEMPERATURE,
      max_tokens=MAX_TOKENS,
  )

  print("\n--- Chat roles output ---")
  print(out_chat["choices"][0]["message"]["content"])


In [ ]:
ask_llm_plain_prompt(cpu_llm)
ask_llm_chat_prompt(cpu_llm)

#### 1.1.2. GPU demo


In [ ]:
# --- 1) Download a larger Llama GGUF model ---
model_path_gpu = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
    filename="llama-2-7b-chat.Q4_K_M.gguf"
  )
print("GPU model path:", model_path_gpu)


# --- 2) Load with llama.cpp ---

llm_gpu = Llama(
    model_path=model_path_gpu,
    n_ctx=N_CTX,
    n_gpu_layers=-1,   # offload all layers to GPU
    verbose=False
)

print("🚀 Llama GPU loaded.")


In [ ]:
# --- 3) Inference ---
ask_llm_plain_prompt(llm_gpu)
ask_llm_chat_prompt(llm_gpu)

### 1.2. Inference via external API

In this section, we will use the OpenAI API as an example. You will learn how to work with Yandex Cloud AI Studio in the next lessons.

#### 1.2.1 Setup
First, [generate](https://platform.openai.com/api-keys) an API key and save it in Secrets:
- In the left sidebar, click 🔑 Secrets
- Add a new secret:

```
Name: OPENAI_API_KEY
Value: sk-...
```


Or just paste it below.

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

#### 1.2.2 Call OpenAI with the same prompts and settings


In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-5-nano"

def call_openai(messages, model=MODEL):
    resp = client.responses.create(
        model=model,
        input=messages,
    )
    return resp.output_text


print("\n--- OpenAI Chat roles output ---")
print(call_openai(MESSAGES_PROMPT))


## 2. Key takeaways about LLM through examples

### 2.1. Hallucinations

The model may confidently generate false information.

In [ ]:
def show(title, text):
  print(f"\n=== {title} ===\n{text}\n")

In [ ]:
messages = [
    {"role":"system","content":"You are a helpful Python assistant."},
    {"role":"user","content":
     "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Hallucination risk", text)

However, `client.responses.create_json_schema()` does not exist.

Hallucinations can be reduced with clear instructions.

In [ ]:
messages = [
    {"role":"system","content":
     "If you are not sure, say 'I don't know'. Do not invent APIs."},
    {"role":"user","content":
     "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Cautious mode", text)

### 2.2. LLM Strengths

1.  Language tasks such as summarization and paraphring

In [ ]:
paragraph = """
Large language models predict the next token rather than verify facts.
They generate fluent text and summaries but may hallucinate.
They struggle with exact counting and up-to-date knowledge without tools.
"""

messages = [
    {"role":"system","content":"Summarize for a lecture slide in a few words"},
    {"role":"user","content":paragraph}
]

text = call_openai(messages)
show("Summarization", text)

2. Generating standard structures

In [ ]:
messages = [
    {"role":"system","content":"Return ONLY valid JSON."},
    {"role":"user","content":
     "Generate JSON with keys: strengths (3 items), weaknesses (2 items) of LLM."}
]

text = call_openai(messages)
show("Structured output", text)

### 2.3. LLM Weaknesses

1. Counting characters

In [ ]:
s = "a"*37 + "b"*41 + "c"*19 + "b"*5

messages = [
    {"role":"system","content":"Answer with ONE integer only."},
    {"role":"user","content":f"How many characters are in this string?\n{s}"}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Counting characters task", text)

print("Ground truth:", len(s))


2. Exact arithmetic

In [ ]:
expr = "17*19*23/9"

messages = [
    {"role":"system","content":"Compute exactly. Output only the number."},
    {"role":"user","content":f"Compute {expr}"}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Math", text)

print("Ground truth:", eval(expr))


3. Up-to-date knowledge


In [ ]:
messages = [
    {"role":"system","content":"You do not have internet access."},
    {"role":"user","content":"What is the current Bitcoin price right now?"}
]

text = call_openai(messages)
show("Up-to-date info", text)


4. Without external context, the model cannot provide exact quotes.

In [ ]:
messages = [
    {"role":"user","content":
     "Give the exact first paragraph of 'Harry Potter and the Philosopher's Stone'."}
]

text = call_openai(messages)
show("Quote without context", text)


book_excerpt = """
Mr and Mrs Dursley, of number four, Privet Drive, were proud to say
that they were perfectly normal, thank you very much.
"""

messages = [
    {"role":"system","content":
     "Quote only from the provided text."},
    {"role":"user","content":
     f"Text:\n{book_excerpt}\n\nQuestion: What does the first sentence say?"}
]

text = call_openai(messages)
show("Quote WITH context", text)